# Creación y Evaluación de Modelos

Tras la preparación y transformación de los datos, se procederá a la fase de **modelado**.  
En esta etapa se implementarán diferentes algoritmos de clasificación para el análisis de riesgo crediticio, con el objetivo de comparar su desempeño y seleccionar el más adecuado.

## Pasos principales

1. **Selección de algoritmos**  
   - Modelos lineales (Regresión logística).  
   - Árboles de decisión y Random Forest.  
   - Métodos basados en boosting (XGBoost, LightGBM).  
   - Redes neuronales simples para comparación.

2. **Entrenamiento de modelos**  
   - Se entrenarán utilizando el conjunto de entrenamiento balanceado (SMOTE aplicado).  
   - Se ajustarán hiperparámetros mediante validación cruzada.

3. **Evaluación de desempeño**  
   - Métricas principales: Accuracy, Precision, Recall, F1-Score, AUC-ROC.  
   - Matriz de confusión para analizar errores de clasificación.  
   - Curvas ROC y PR para comparar modelos.

4. **Selección del modelo final**  
   - Se elegirá el modelo con mejor equilibrio entre métricas.  
   - Se documentará el proceso de selección y justificación técnica.

## Resultado esperado

Al finalizar esta etapa se contará con:
- Un **modelo entrenado y validado** listo para ser desplegado.  
- Reportes de métricas y visualizaciones que respalden la elección del modelo.  
- Un pipeline reproducible que permita repetir el proceso con nuevos datos.


In [47]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
# Redes neuronales -- Para modelos puede ser util pero explicativamente son mas complejos
# from tensorflow import keras
# from tensorflow.keras import layers
from scripts.pipeline import run_pipeline

In [48]:
run_pipeline("data/german_credit_data.csv", "data/clean_credit.csv")

Starting pipeline...
Data extracted.
Data transformed.
Data successfully saved to data/clean_credit.csv
Pipeline completed successfully.


In [49]:
# Cargar el dataset limpio
df = pd.read_csv("data/clean_credit.csv")
df

,Age,Job,Credit amount,Duration,Risk,Sex_female,Sex_male,Housing_free,Housing_own,Housing_rent,...,Checking account_rich,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others,duration_per_year
0,2.266667,2,-0.441354,-1.00,1,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.083333
1,-0.733333,2,1.393114,2.50,0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.208333
2,1.066667,1,-0.085739,-0.50,1,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,-0.041667
3,0.800000,2,2.133883,2.00,1,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.166667
4,1.333333,2,0.978421,0.50,0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.041667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.133333,1,-0.223842,-0.50,1,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.041667
996,0.466667,3,0.589815,1.00,1,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.083333
997,0.333333,2,-0.581375,-0.50,1,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.041667
998,-0.666667,2,-0.182027,2.25,0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.187500


In [50]:
# Separar características y variable objetivo
X = df.drop("Risk", axis=1)
y = df["Risk"]

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [52]:
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

## **Regresion Logistica**

In [53]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((800, 25), (800,), (200, 25), (200,))

In [89]:
reglog = LogisticRegression()

In [90]:
param_grid = {
    'C': [0.1, 0.2, 0.5, 0.8, 1],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']}

In [91]:
# definiremos un objeto de la clase KFold() para realizar la validación cruzada, especificando 10 subconjuntos con el parámetro n_splits y reordenando el conjunto de datos en cada iteración con el parámetro shuffle

kfold = KFold(n_splits=10, shuffle=True, random_state=0)

In [92]:
grid = GridSearchCV(estimator=reglog, param_grid=param_grid, cv=kfold, n_jobs=-1, verbose=2)

In [93]:
# Entrenar el modelo
grid.fit(X_train, y_train)

Fitting 10 folds for each of 20 candidates, totalling 200 fits


GridSearchCV(cv=KFold(n_splits=10, random_state=0, shuffle=True),
             estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': [0.1, 0.2, 0.5, 0.8, 1], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear', 'saga']},
             verbose=2)

In [94]:
print("Mejores parámetros: {}".format(grid.best_params_)) 

Mejores parámetros: {'C': 0.2, 'penalty': 'l1', 'solver': 'saga'}


In [95]:
mejor_modelo = grid.best_estimator_

In [61]:
pd.DataFrame(zip(X_train.columns, mejor_modelo.coef_[0]),
             columns=["Variable","Coeficiente"]
            ).sort_values("Coeficiente",ascending=False)

,Variable,Coeficiente
7,Housing_own,0.515863
21,Purpose_radio/TV,0.446900
0,Age,0.370634
5,Sex_male,0.175535
13,Checking account_little,0.000000
23,Purpose_vacation/others,0.000000
22,Purpose_repairs,0.000000
20,Purpose_furniture/equipment,0.000000
19,Purpose_education,0.000000
18,Purpose_domestic appliances,0.000000


En regresión logística, la magnitud de cada coeficiente puede ser utilizado para determinar la relevancia de la variable en la clasificación que realiza el modelo

In [96]:
y_pred_logg_1 = mejor_modelo.predict(X_test)
y_proba_logg_1 = mejor_modelo.predict_proba(X_test)[:,1]

In [63]:
confusion_matrix(y_test, y_pred_logg_1)

array([[ 10,  49],
       [  8, 133]], dtype=int64)

In [64]:
print(classification_report(y_test, y_pred_logg_1))

              precision    recall  f1-score   support

           0       0.56      0.17      0.26        59
           1       0.73      0.94      0.82       141

    accuracy                           0.71       200
   macro avg       0.64      0.56      0.54       200
weighted avg       0.68      0.71      0.66       200



- El modelo está desbalanceado en su desempeño: favorece mucho la clase mayoritaria (1) y falla en la minoritaria (0).
- Esto suele pasar cuando el dataset está desbalanceado (más ejemplos de una clase que de otra).
- Aunque la exactitud es aceptable (71%), el bajo recall de la clase 0 indica que el modelo no es confiable para detectar correctamente los casos de bajo riesgo.

Próximos pasos:
- Balancear las clases
- Usar técnicas como SMOTE, undersampling o class_weight="balanced" en LogisticRegression o RandomForestClassifier.

El modelo actual detecta muy bien la clase 1 pero ignora la clase 0. Para un caso de riesgo crediticio, esto es crítico porque podríamos estar rechazando clientes buenos o aceptando demasiados malos.


In [65]:
reglog_2 = LogisticRegression(class_weight="balanced", max_iter=1000)

param_grid_2 = {
    'C': [0.1, 0.2, 0.5, 0.8, 1],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}

grid_2 = GridSearchCV(estimator=reglog_2, param_grid=param_grid_2, cv=kfold, n_jobs=-1, verbose=2)
grid_2.fit(X_train, y_train)
print("Mejores parámetros: {}".format(grid_2.best_params_))

Fitting 10 folds for each of 20 candidates, totalling 200 fits
Mejores parámetros: {'C': 0.2, 'penalty': 'l1', 'solver': 'liblinear'}


Sigue indicando que los mejores parametros son: {'C': 0.2, 'class_weight': None, 'penalty': 'l1', 'solver': 'saga'}

In [66]:
mejor_modelo_2 = grid_2.best_estimator_
y_pred_logg_2 = mejor_modelo_2.predict(X_test)
y_proba_logg_2 = mejor_modelo_2.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred_logg_2))

              precision    recall  f1-score   support

           0       0.40      0.49      0.44        59
           1       0.76      0.69      0.72       141

    accuracy                           0.63       200
   macro avg       0.58      0.59      0.58       200
weighted avg       0.66      0.63      0.64       200



Al incluir class_weight="balanced" el modelo mejoro el recall de la clase 0 (pasó de 0.17 a 0.49). Eso significa que ahora detecta casi la mitad de los casos de bajo riesgo, aunque sacrificó algo de precisión y exactitud global (accuracy bajó de 0.71 a 0.63).

## **RandomForestClassifier and DecisionTreeClassifier**

In [67]:
# Modelo de árbol de decisión

Tree_Classifier = DecisionTreeClassifier(random_state=77)
# Definimos la grilla de parámetros para el modelo de árbol de decisión
param_grid_dt = {
    'criterion': ['gini', 'entropy'], # Define el criterio de división del árbol
    'max_depth': [None, 5, 10, 20], # Define la profundidad máxima del árbol
    'min_samples_split': [2, 5, 10], # Define el número mínimo de muestras necesarias para dividir un nodo
    'min_samples_leaf': [1, 2, 4], # Define el número mínimo de muestras necesarias para ser una hoja
    'class_weight': [None, 'balanced'] # Define el peso de las clases
}


# definiremos un objeto de la clase KFold() para realizar la validación cruzada con 5 subconjuntos de datos:
kfold = KFold(n_splits=5, shuffle=True, random_state=77)

grid_search_dt = GridSearchCV(estimator=Tree_Classifier, param_grid=param_grid_dt, cv=kfold, n_jobs=-1, verbose=1)
grid_search_dt

GridSearchCV(cv=KFold(n_splits=5, random_state=77, shuffle=True),
             estimator=DecisionTreeClassifier(random_state=77), n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced'],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             verbose=1)

In [68]:
grid_search_dt.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


GridSearchCV(cv=KFold(n_splits=5, random_state=77, shuffle=True),
             estimator=DecisionTreeClassifier(random_state=77), n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced'],
                         'criterion': ['gini', 'entropy'],
                         'max_depth': [None, 5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             verbose=1)

In [69]:
print("Mejores parámetros: {}".format(grid_search_dt.best_params_))

Mejores parámetros: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [70]:
mejor_modelo_tree = grid_search_dt.best_estimator_

In [71]:
y_test_pred_tree = mejor_modelo_tree.predict(X_test)
y_proba_tree = mejor_modelo_tree.predict_proba(X_test)[:,1]

In [72]:
confusion_matrix(y_test, y_test_pred_tree)

array([[ 11,  48],
       [ 19, 122]], dtype=int64)

In [73]:
print(classification_report(y_test, y_test_pred_tree))

              precision    recall  f1-score   support

           0       0.37      0.19      0.25        59
           1       0.72      0.87      0.78       141

    accuracy                           0.67       200
   macro avg       0.54      0.53      0.52       200
weighted avg       0.61      0.67      0.63       200



In [74]:
# Modelo de árbol de decisión

Tree_Classifier_2 = DecisionTreeClassifier(random_state=77, class_weight="balanced")
# Definimos la grilla de parámetros para el modelo de árbol de decisión
param_grid_dt_2 = {
    'criterion': ['gini', 'entropy'], # Define el criterio de división del árbol
    'max_depth': [None, 5, 10, 20], # Define la profundidad máxima del árbol
    'min_samples_split': [2, 5, 10], # Define el número mínimo de muestras necesarias para dividir un nodo
    'min_samples_leaf': [1, 2, 4], # Define el número mínimo de muestras necesarias para ser una hoja
    'class_weight': [None, 'balanced'] # Define el peso de las clases
}

grid_search_dt_2 = GridSearchCV(estimator=Tree_Classifier_2, param_grid=param_grid_dt_2, cv=kfold, n_jobs=-1, verbose=1)
grid_search_dt_2.fit(X_train, y_train)
print("Mejores parámetros: {}".format(grid_search_dt_2.best_params_))

Fitting 5 folds for each of 144 candidates, totalling 720 fits


Mejores parámetros: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [75]:
mejor_modelo_tree_2 = grid_search_dt_2.best_estimator_
y_test_pred_2_tree = mejor_modelo_tree_2.predict(X_test)
y_proba_tree_2 = mejor_modelo_tree_2.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_test_pred_2_tree))

              precision    recall  f1-score   support

           0       0.37      0.19      0.25        59
           1       0.72      0.87      0.78       141

    accuracy                           0.67       200
   macro avg       0.54      0.53      0.52       200
weighted avg       0.61      0.67      0.63       200



In [76]:
# Definimos el modelo base con class_weight="balanced"
rf_clf = RandomForestClassifier(random_state=77, class_weight="balanced")

# Definimos la grilla de parámetros para Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],        # número de árboles
    'criterion': ['gini', 'entropy'],       # criterio de división
    'max_depth': [None, 5, 10, 20],         # profundidad máxima
    'min_samples_split': [2, 5, 10],        # mínimo de muestras para dividir un nodo
    'min_samples_leaf': [1, 2, 4]           # mínimo de muestras en una hoja
}

# Validación cruzada
kfold = KFold(n_splits=10, shuffle=True, random_state=0)

# GridSearchCV
grid_search_rf = GridSearchCV(estimator=rf_clf, 
                            param_grid=param_grid_rf, 
                            cv=kfold, 
                            n_jobs=-1, 
                            verbose=1)

# Entrenamiento
grid_search_rf.fit(X_train, y_train)

# Mejor modelo
print("Mejores parámetros:", grid_search_rf.best_params_)
best_rf = grid_search_rf.best_estimator_

# Predicciones
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:,1]

# Reporte de clasificación
print(classification_report(y_test, y_pred_rf))


Fitting 10 folds for each of 216 candidates, totalling 2160 fits
Mejores parámetros: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
              precision    recall  f1-score   support

           0       0.46      0.22      0.30        59
           1       0.73      0.89      0.81       141

    accuracy                           0.69       200
   macro avg       0.60      0.56      0.55       200
weighted avg       0.65      0.69      0.66       200



## **XGBClassifier**

In [77]:
# Modelo XGBoost
xgb_clf = XGBClassifier(random_state=77, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100, 200, 300], # Definicion de 
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_child_weight': [1, 2, 3],
    'gamma': [0, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.2],
    'reg_lambda': [0, 0.1, 0.2]
}
# Validación cruzada
kfold = KFold(n_splits=2, shuffle=True, random_state=0)

grid_search_xgb = GridSearchCV(estimator=xgb_clf, 
                            param_grid=param_grid_xgb, 
                            cv=kfold,  
                            n_jobs=-1,  
                            verbose=1)

grid_search_xgb.fit(X_train, y_train)
print("Mejores parámetros:", grid_search_xgb.best_params_)
best_xgb = grid_search_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred_xgb))

Fitting 2 folds for each of 8748 candidates, totalling 17496 fits


C:\Users\masso\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:200: UserWarning: [18:58:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Mejores parámetros: {'colsample_bytree': 1.0, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_weight': 2, 'n_estimators': 200, 'reg_alpha': 0.1, 'reg_lambda': 0, 'subsample': 0.8}
              precision    recall  f1-score   support

           0       0.57      0.20      0.30        59
           1       0.74      0.94      0.82       141

    accuracy                           0.72       200
   macro avg       0.65      0.57      0.56       200
weighted avg       0.69      0.72      0.67       200



## **Comparativa de modelos**

In [97]:
from sklearn.metrics import roc_auc_score, classification_report

# Evaluación del modelo de regresión logística
print(classification_report(y_test, y_pred_logg_1))
print("ROC AUC:", roc_auc_score(y_test, y_proba_logg_1))

# Guardando variables para comparacion recall, precission y ROC AUC
log_reg_recall_log_1 = classification_report(y_test, y_pred_logg_1, output_dict=True)['1']['recall']
log_reg_precision_log_1 = classification_report(y_test, y_pred_logg_1, output_dict=True)['1']['precision']
log_reg_roc_auc_log_1 = roc_auc_score(y_test, y_proba_logg_1)

              precision    recall  f1-score   support

           0       0.56      0.17      0.26        59
           1       0.73      0.94      0.82       141

    accuracy                           0.71       200
   macro avg       0.64      0.56      0.54       200
weighted avg       0.68      0.71      0.66       200

ROC AUC: 0.6043995672556797


In [81]:
# Evaluación del modelo de regresión logística con class_weight="balanced"
print(classification_report(y_test, y_pred_logg_2))
print("ROC AUC:", roc_auc_score(y_test, y_proba_logg_2))

# Guardando variables para comparacion recall, precission y ROC AUC
log_reg_recall_log_2 = classification_report(y_test, y_pred_logg_2, output_dict=True)['1']['recall']
log_reg_precision_log_2 = classification_report(y_test, y_pred_logg_2, output_dict=True)['1']['precision']
log_reg_roc_auc_log_2 = roc_auc_score(y_test, y_proba_logg_2)

              precision    recall  f1-score   support

           0       0.40      0.49      0.44        59
           1       0.76      0.69      0.72       141

    accuracy                           0.63       200
   macro avg       0.58      0.59      0.58       200
weighted avg       0.66      0.63      0.64       200

ROC AUC: 0.6011539848539488


In [83]:

# Evaluación del modelo de árbol de decisión
print(classification_report(y_test, y_test_pred_tree))
print("ROC AUC:", roc_auc_score(y_test, y_proba_tree))

# Guardando variables para comparacion recall, precission y ROC AUC
log_reg_recall_tree_1 = classification_report(y_test, y_test_pred_tree, output_dict=True)['1']['recall']
log_reg_precision_tree_1 = classification_report(y_test, y_test_pred_tree, output_dict=True)['1']['precision']
log_reg_roc_auc_tree_1 = roc_auc_score(y_test, y_proba_tree)

              precision    recall  f1-score   support

           0       0.37      0.19      0.25        59
           1       0.72      0.87      0.78       141

    accuracy                           0.67       200
   macro avg       0.54      0.53      0.52       200
weighted avg       0.61      0.67      0.63       200

ROC AUC: 0.5432143286452699


In [86]:
# Evaluación del modelo de árbol de decisión con class_weight="balanced"
print(classification_report(y_test, y_test_pred_2_tree))
print("ROC AUC:", roc_auc_score(y_test, y_proba_tree_2))

# Guardando variables para comparacion recall, precission y ROC AUC
log_reg_recall_tree_2 = classification_report(y_test, y_test_pred_2_tree, output_dict=True)['1']['recall']
log_reg_precision_tree_2 = classification_report(y_test, y_test_pred_2_tree, output_dict=True)['1']['precision']
log_reg_roc_auc_tree_2 = roc_auc_score(y_test, y_proba_tree_2)

              precision    recall  f1-score   support

           0       0.37      0.19      0.25        59
           1       0.72      0.87      0.78       141

    accuracy                           0.67       200
   macro avg       0.54      0.53      0.52       200
weighted avg       0.61      0.67      0.63       200

ROC AUC: 0.5432143286452699


In [87]:

# Evaluación del modelo de Random Forest
print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_proba_rf))

# Guardando variables para comparacion recall, precission y ROC AUC
log_reg_recall_rf_1 = classification_report(y_test, y_pred_rf, output_dict=True)['1']['recall']
log_reg_precision_rf_1 = classification_report(y_test, y_pred_rf, output_dict=True)['1']['precision']
log_reg_roc_auc_rf_1 = roc_auc_score(y_test, y_proba_rf)

              precision    recall  f1-score   support

           0       0.46      0.22      0.30        59
           1       0.73      0.89      0.81       141

    accuracy                           0.69       200
   macro avg       0.60      0.56      0.55       200
weighted avg       0.65      0.69      0.66       200

ROC AUC: 0.6465921384781824


In [88]:
# Evaluación del modelo de XGBoost|
print(classification_report(y_test, y_pred_xgb))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

# Guardando variables para comparacion recall, precission y ROC AUC
log_reg_recall_xgb_1 = classification_report(y_test, y_pred_xgb, output_dict=True)['1']['recall']
log_reg_precision_xgb_1 = classification_report(y_test, y_pred_xgb, output_dict=True)['1']['precision']
log_reg_roc_auc_xgb_1 = roc_auc_score(y_test, y_proba_xgb)

              precision    recall  f1-score   support

           0       0.57      0.20      0.30        59
           1       0.74      0.94      0.82       141

    accuracy                           0.72       200
   macro avg       0.65      0.57      0.56       200
weighted avg       0.69      0.72      0.67       200

ROC AUC: 0.6688303882678207


In [103]:
# Comparación de modelos 

# Comparacion para 1
model_comparison = pd.DataFrame({
    'Modelo': ['Logistic Regression', 'Logistic Regression (Balanced)', 'Decision Tree', 'Decision Tree (Balanced)', 'Random Forest', 'XGBoost'],
    'Recall': [log_reg_recall_log_1, log_reg_recall_log_2, log_reg_recall_tree_1, log_reg_recall_tree_2, log_reg_recall_rf_1, log_reg_recall_xgb_1],
    'Recall_0': [classification_report(y_test, y_pred_logg_1, output_dict=True)['0']['recall'], classification_report(y_test, y_pred_logg_2, output_dict=True)['0']['recall'], classification_report(y_test, y_test_pred_tree, output_dict=True)['0']['recall'], classification_report(y_test, y_test_pred_2_tree, output_dict=True)['0']['recall'], classification_report(y_test, y_pred_rf, output_dict=True)['0']['recall'], classification_report(y_test, y_pred_xgb, output_dict=True)['0']['recall']],
    'Precision': [log_reg_precision_log_1, log_reg_precision_log_2, log_reg_precision_tree_1, log_reg_precision_tree_2, log_reg_precision_rf_1, log_reg_precision_xgb_1],
    'Precision_0': [classification_report(y_test, y_pred_logg_1, output_dict=True)['0']['precision'], classification_report(y_test, y_pred_logg_2, output_dict=True)['0']['precision'], classification_report(y_test, y_test_pred_tree, output_dict=True)['0']['precision'], classification_report(y_test, y_test_pred_2_tree, output_dict=True)['0']['precision'], classification_report(y_test, y_pred_rf, output_dict=True)['0']['precision'], classification_report(y_test, y_pred_xgb, output_dict=True)['0']['precision']],
    'ROC AUC': [log_reg_roc_auc_log_1, log_reg_roc_auc_log_2, log_reg_roc_auc_tree_1, log_reg_roc_auc_tree_2, log_reg_roc_auc_rf_1, log_reg_roc_auc_xgb_1]

})

In [104]:
model_comparison

,Modelo,Recall,Recall_0,Precision,Precision_0,ROC AUC
0,Logistic Regression,0.943262,0.169492,0.730769,0.555556,0.604400
1,Logistic Regression (Balanced),0.687943,0.491525,0.763780,0.397260,0.601154
2,Decision Tree,0.865248,0.186441,0.717647,0.366667,0.543214
3,Decision Tree (Balanced),0.865248,0.186441,0.717647,0.366667,0.543214
4,Random Forest,0.893617,0.220339,0.732558,0.464286,0.646592
5,XGBoost,0.936170,0.203390,0.737430,0.571429,0.668830


# **Analisis** 

Codificar la variable objetivo "Risk" utilizando LabelEncoder, donde "good" se codifica como 0 y "bad" como 1
Aunque Logistic Regression balanceado presenta un mejor equilibrio entre clases, seleccioné XGBoost debido a su mayor capacidad discriminativa (ROC AUC). Posteriormente, ajustaría el threshold para lograr un balance adecuado entre recall de la clase bad y aprobación de clientes good.

In [106]:
# Definir pesos
w_recall = 0.5
w_roc = 0.3
w_precision = 0.2

model_comparison['score'] = (
    model_comparison['Recall'] * w_recall +
    model_comparison['ROC AUC'] * w_roc +
    model_comparison['Precision'] * w_precision
)

best_row = model_comparison.sort_values(by='score', ascending=False).iloc[0]
best_model_name = best_row['Modelo']

model_dict = {
    'XGBoost': best_xgb,
    'Random Forest': best_rf,
    'Decision Tree (Balanced)': mejor_modelo_tree_2,
    'Decision Tree': mejor_modelo_tree,
    'Logistic Regression': mejor_modelo,
    'Logistic Regression (Balanced)': mejor_modelo_2
}

best_model = model_dict[best_model_name]

print("El mejor modelo es:", best_model_name)
print("Score:", best_row['score'])

El mejor modelo es: XGBoost
Score: 0.816220256382878


# 📦 Empaquetado del Modelo de Credit Risk

Seleccionar y empaquetar el mejor modelo de Machine Learning para **Credit Risk Modeling**, basado en métricas clave que reflejan tanto el desempeño estadístico como el impacto en negocio.

---

## 🧠 Criterios de Selección del Modelo

Debido a la naturaleza desbalanceada del problema (clientes *good* vs *bad*), no se utiliza **accuracy** como métrica principal. En su lugar, se priorizan las siguientes métricas:

### 🏆 ROC AUC
- Mide la capacidad del modelo para diferenciar entre clientes buenos y malos.
- Es independiente del umbral de clasificación.
- Permite comparar modelos de manera robusta.

### 🔍 Recall (Clase "Bad")
- Mide la capacidad del modelo para identificar correctamente a los clientes riesgosos.
- Es crítica en el contexto de riesgo crediticio, ya que reduce la probabilidad de aprobar clientes con alta probabilidad de default.

### ⚖️ Precision (Clase "Bad")
- Mide la proporción de clientes correctamente identificados como riesgosos.
- Ayuda a evitar rechazar clientes que en realidad son buenos.

---

## 🧪 Proceso de Evaluación

1. Se entrenaron múltiples modelos:
   - Logistic Regression
   - Random Forest
   - Decision Tree
   - XGBoost

2. Se utilizó:
   - `class_weight="balanced"` para mitigar el desbalance de clases.
   - Validación cruzada y/o conjunto de prueba.

3. Para cada modelo se calcularon:
   - ROC AUC
   - Recall (clase bad)
   - Precision (clase bad)

4. Se compararon los resultados y se seleccionó el modelo que:
   - Maximiza ROC AUC
   - Tiene alto Recall en clase "bad"
   - Mantiene un Precision aceptable

---

## 📊 Selección del Mejor Modelo

El modelo seleccionado es aquel que logra el mejor balance entre:

- Alta capacidad discriminativa (ROC AUC)
- Alta detección de clientes riesgosos (Recall)
- Bajo nivel de falsos positivos (Precision)

---

## 📦 Empaquetado del Modelo

Una vez seleccionado el mejor modelo:

### ✔️ Se guarda el modelo entrenado en la carpeta model
```python
import joblib

joblib.dump(best_model, "credit_risk_model.pkl")

In [107]:
# Empaquetar el mejor modelo
joblib.dump(best_model, "best_model.pkl")

['best_model.pkl']